<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor: experiment notebook

This notebook compares five formulations for the periodic two-well **committor probability** PDE, following the methods in [Tepakbong et al. (2026)](https://arxiv.org/abs/2607.19167).

**Start here:** choose the device and save location in the next section before running the notebook. Then run the cells from top to bottom through Setup, CPU preflight, code check, CPU preparation, and training. Choose the matching Colab runtime before Setup (`"cuda"` → GPU, `"cpu"` → CPU). Setup does not connect to Drive. Preparation and training keep the selected runtime; reference and Monte Carlo preparation use CPU either way.

Each new preparation draws and records fresh training and Monte Carlo seeds. Training reads the seeds and results folder made by that preparation.

[Experiment guide: full derivation, loss definitions, saved-file map, and references](https://github.com/n2pakbong/boundary-adapted-pinns/blob/main/experiments/committor/periodic_two_well/README.md).

## Start here: choose hardware and where results are saved

Change the next code cell before running Setup.

- **Default: Colab storage.** Leave `SAVE_TO_DRIVE = False`. Nothing connects to Google Drive. Results are written under `LOCAL_RESULTS_DIR`. The inspection cell downloads a ZIP of the full run when `DOWNLOAD_LOCAL_RESULTS = True`. Download it before closing or resetting Colab; local files are temporary.
- **Google Drive.** Set `SAVE_TO_DRIVE = True`. Results are written under `DRIVE_RESULTS_DIR`. Colab asks to connect to Drive only when preparation first needs to save there, or when a later cell needs to reopen that Drive run.
- **Each run gets its own folder.** Preparation creates a new uniquely named folder under the selected directory. It contains the settings, reference and Monte Carlo data, training histories and checkpoints, metrics, prediction/error arrays, and figures. The notebook keeps the folder path and passes it to training and inspection automatically. You never copy or paste a folder path.
- **After a runtime reset.** Drive runs can be found again automatically. If several unfinished runs match, choose the run by its number. Local files do not survive a reset.

In [ ]:
# Choose the device and result destination before Setup.
DEVICE = "cuda"       # Use "cpu" or "cuda" (or "cuda:0"); match the Colab runtime.
DTYPE = "float64"     # Use "float32" or "float64".

# Keep False for temporary Colab storage; set True only to save in Google Drive.
SAVE_TO_DRIVE = False
LOCAL_RESULTS_DIR = "/content/periodic_committor_runs"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor"
DOWNLOAD_LOCAL_RESULTS = True

import secrets
SEED = secrets.randbelow(2**31)  # Code-check seed; each saved run draws new seeds.

## 1. The mathematical problem

Consider overdamped Langevin dynamics on the flat torus $\mathbb T^2=[0,1)^2$:

$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t \pmod 1,
\qquad
V(x_1,x_2)=a\left[1-\cos\!\left(4\pi(x_1-\tfrac14)\right)\right]
+b\left[1-\cos(2\pi x_2)\right].
$$

For the default settings, $A$ and $B$ are disjoint periodic disks, centered at $(1/4,0)$ and $(3/4,0)$. The **forward committor** is

$$
q(x)=\mathbb P_x(\tau_B<\tau_A),
$$

the probability that a trajectory started at $x$ reaches $B$ before $A$. It solves

$$
\mathcal L q
=-\nabla V\cdot\nabla q+\varepsilon\Delta q=0
\quad\text{in }D=\mathbb T^2\setminus(A\cup B),
$$

with $q=0$ on $\partial A$ and $q=1$ on $\partial B$. Thus this notebook studies a hitting probability, not a mean escape time.

The starting values are $a=1$, $b=0.6$, $\varepsilon=0.4$, and periodic chordal radius $r=0.5$. The finite-difference solution is a numerical target for error measurement, not exact ground truth.

This notebook supports one problem family: a flat unit torus with two equal-radius periodic disks and the cosine potential shown above. You can change the potential amplitudes, diffusion strength, disk centers, and common radius below. The torus, disk shape, potential formula, and cosine phase centers stay fixed. Moving a disk does not move a potential well. The runner rejects disks that overlap or touch. A different domain, disk geometry, or potential formula requires code changes to the problem implementation and its samplers.

### Change the potential and the two boundary sets

The domain stays the flat unit torus: each coordinate wraps around at 0 and 1. For readability, enter each center coordinate in the range from 0 up to (but not including) 1; for example, 0.25 is one quarter of the way around the torus. The coordinates wrap periodically. Set A is the source, where the committor is 0. Set B is the target, where it is 1. Both use the same periodic chordal radius. If the two closed disks overlap or touch, the run stops with an error. The radius must lie strictly between 0 and 1.

`POTENTIAL_A` and `POTENTIAL_B` are the amplitudes in the displayed cosine formula; `DIFFUSION_EPSILON` is the noise strength and must be positive. Moving a disk center does not move a cosine well. This interface cannot change the dimension, domain, disk shape, or potential formula.


In [ ]:
# Edit these ordinary Python values to change the example problem.
POTENTIAL_A = 1.0
POTENTIAL_B = 0.6
DIFFUSION_EPSILON = 0.4
SOURCE_CENTER = (0.25, 0.0)
TARGET_CENTER = (0.75, 0.0)
COMMON_CHORDAL_RADIUS = 0.5


## The network

Edit width, depth, activation, initialization, and bias settings at the top of the next cell. The periodic feature map and model class are directly below; change them if you want a different architecture.


In [ ]:
import torch
from torch import nn

# Network controls: edit these values before the setup/check cell.
NETWORK_WIDTH = 64
NETWORK_DEPTH = 2
ACTIVATION_NAME = "tanh"          # smooth choices: tanh, softplus, silu, gelu
INITIALIZATION_NAME = "pytorch"    # pytorch, xavier_normal, or xavier_uniform
INITIALIZATION_GAIN = 1.0  # Xavier initializers only
BIAS_INITIALIZATION = "pytorch"   # pytorch, zeros, or constant
BIAS_INITIALIZATION_VALUE = 0.0
USE_BIAS = True


def make_activation():
    if ACTIVATION_NAME == "tanh":
        return nn.Tanh()
    if ACTIVATION_NAME == "softplus":
        return nn.Softplus()
    if ACTIVATION_NAME == "silu":
        return nn.SiLU()
    if ACTIVATION_NAME == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown ACTIVATION_NAME: {ACTIVATION_NAME}")


class PeriodicCommittorNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        if NETWORK_WIDTH < 1 or NETWORK_DEPTH < 1:
            raise ValueError("NETWORK_WIDTH and NETWORK_DEPTH must be positive.")
        layers = [nn.Linear(4, NETWORK_WIDTH, bias=USE_BIAS), make_activation()]
        for _ in range(NETWORK_DEPTH - 1):
            layers.extend((nn.Linear(NETWORK_WIDTH, NETWORK_WIDTH, bias=USE_BIAS), make_activation()))
        layers.append(nn.Linear(NETWORK_WIDTH, 1, bias=USE_BIAS))
        self.layers = nn.Sequential(*layers)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if not isinstance(module, nn.Linear):
            return
        if INITIALIZATION_NAME == "pytorch":
            # Keep nn.Linear's default weight initialization.
            pass
        elif INITIALIZATION_NAME == "xavier_normal":
            nn.init.xavier_normal_(module.weight, gain=INITIALIZATION_GAIN)
        elif INITIALIZATION_NAME == "xavier_uniform":
            nn.init.xavier_uniform_(module.weight, gain=INITIALIZATION_GAIN)
        else:
            raise ValueError(f"Unknown INITIALIZATION_NAME: {INITIALIZATION_NAME}")
        if module.bias is not None:
            if BIAS_INITIALIZATION == "zeros":
                nn.init.zeros_(module.bias)
            elif BIAS_INITIALIZATION == "constant":
                nn.init.constant_(module.bias, BIAS_INITIALIZATION_VALUE)
            elif BIAS_INITIALIZATION != "pytorch":
                raise ValueError(f"Unknown BIAS_INITIALIZATION: {BIAS_INITIALIZATION}")

    def forward(self, x):
        phase = 2.0 * torch.pi * x
        features = torch.cat((torch.sin(phase), torch.cos(phase)), dim=-1)
        return self.layers(features).squeeze(-1)


NOTEBOOK_MODEL_SETTINGS = {
    "input_features": ["sin(2*pi*x1)", "sin(2*pi*x2)", "cos(2*pi*x1)", "cos(2*pi*x2)"],
    "width": NETWORK_WIDTH,
    "depth": NETWORK_DEPTH,
    "activation": ACTIVATION_NAME,
    "initialization": INITIALIZATION_NAME,
    "initialization_gain": INITIALIZATION_GAIN,
    "bias_initialization": BIAS_INITIALIZATION,
    "bias_initialization_value": BIAS_INITIALIZATION_VALUE,
    "use_bias": USE_BIAS,
}

print(PeriodicCommittorNetwork())


### Interior sampling

This function proposes points uniformly from `[0, 1)²` and rejects points inside A or B, so accepted points are uniform by area on the punctured torus D. It supplies both PDE collocation points and Monte Carlo anchors. Edit the function to use another distribution; every point must remain in D. A nonuniform law changes the empirical averages unless you add importance weights. Boundary points use a separate sampler: uniform radial angles with arclength weights in boundary losses and Nitsche integrals.

In [ ]:
def sample_training_points(problem, n, *, device, dtype, generator=None):
    # Uniform rejection sampling on D = [0, 1)^2 \\ (A union B).
    accepted = []
    remaining = n
    for _ in range(100):
        proposal = torch.rand(
            max(2 * remaining, 32), 2,
            device=device, dtype=dtype, generator=generator,
        )
        inside_D = problem.domain_mask(proposal)
        take = proposal[inside_D][:remaining]
        if len(take):
            accepted.append(take)
            remaining -= len(take)
        if remaining == 0:
            return torch.cat(accepted, dim=0)
    raise RuntimeError("Could not sample enough points in D; check the geometry.")

## 2. What runs, and what uses compute

| Stage | Work | Compute |
|---|---|---|
| Setup | Install code and run tests in a child process with CUDA hidden | CPU |
| Preflight | Tiny one-epoch run of all five methods | CPU |
| Preparation | Finite-difference references and shared Monte Carlo labels | CPU; saves to the folder selected at the start |
| Training | Selected networks and solution-error evaluation | The configured device |
| Inspect/evaluate | Saved curves, losses, errors; checkpoint evaluation | CPU |

Choose `DEVICE` and the matching Colab runtime in the opening start section. Keep that runtime through preparation and training: preparation uses CPU either way, and switching later resets the notebook's live Python state. Model, sampler, loss, optimizer, schedule, training, and reporting settings are editable in the cells below. Each preparation uses fresh recorded seeds.

### Pick the methods to compare

Edit the Python list in the next cell. It accepts these names: `simple_pinn`, `boundary_adapted`, `over_enforced`, `under_enforced`, and `variational_nitsche`. Their trial functions and losses are described above. The run trains only the methods in this list; the small preflight still checks all five methods.

In [ ]:
METHODS_TO_RUN = [
    "simple_pinn",
    "boundary_adapted",
    "over_enforced",
    "under_enforced",
    "variational_nitsche",
]


## 3. Setup and CPU tests

Use the `DEVICE` value from the opening start section and choose the matching Colab runtime, then run Setup. It installs the repository's `main` branch, reloads the package in this kernel, and runs tests with CUDA hidden in a separate process. Setup does not mount Drive or train.

With `SAVE_TO_DRIVE = False`, Drive is never connected. With `True`, Colab connects when preparation first creates a run folder, or when a later cell must reopen that Drive run. Keep the runtime through preflight, preparation, and training; changing runtimes resets the live Python state.

In [ ]:
# Install one repository revision and run its CPU tests.
from pathlib import Path
import importlib
import os
import subprocess
import sys

from google.colab import drive

def ensure_drive_mounted():
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.is_dir():
        return
    drive.mount("/content/drive", force_remount=False)
    if not drive_root.is_dir():
        raise RuntimeError("Google Drive did not mount. Reconnect Colab or choose local output.")

def run_output_parent():
    if SAVE_TO_DRIVE:
        ensure_drive_mounted()
        return Path(DRIVE_RESULTS_DIR).expanduser()
    return Path(LOCAL_RESULTS_DIR).expanduser()


def resolve_run_root(required_file, purpose, unfinished_only=False):
    current_root = globals().get("RUN_ROOT")
    if current_root:
        current_path = Path(current_root)
        current_run_is_usable = current_path.is_dir() and (current_path / required_file).is_file()
        if unfinished_only:
            current_run_is_usable = (
                current_run_is_usable and not (current_path / "summary.json").is_file()
            )
        if current_run_is_usable:
            return current_path

    output_parent = run_output_parent()
    candidates = [
        path for path in output_parent.glob("periodic_committor_*")
        if path.is_dir() and (path / required_file).is_file()
    ]
    if unfinished_only:
        candidates = [path for path in candidates if not (path / "summary.json").is_file()]
    candidates.sort(key=lambda path: path.stat().st_mtime, reverse=True)

    if not candidates:
        raise FileNotFoundError(
            f"No saved run for {purpose} was found in {output_parent}. "
            "Run the required earlier notebook steps first. Local Colab files are lost "
            "when that runtime is reset."
        )

    if len(candidates) == 1:
        selected = candidates[0]
    else:
        print(f"Choose the run to {purpose}:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        answer = input(f"Run number [1-{len(candidates)}]: ").strip()
        try:
            selection = int(answer)
        except ValueError as error:
            raise ValueError("Enter one of the displayed run numbers.") from error
        if not 1 <= selection <= len(candidates):
            raise ValueError("Choose a run number from the displayed list.")
        selected = candidates[selection - 1]

    globals()["RUN_ROOT"] = str(selected)
    print(f"Using saved run: {selected}")
    return selected


CPU_PREFLIGHT_PASSED = False
NOTEBOOK_CODE_SMOKE_PASSED = False

REPO = Path("/content/boundary-adapted-pinns")
REPO_URL = "https://github.com/n2pakbong/boundary-adapted-pinns.git"
BRANCH = "main"

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"{REPO} exists but is not a Git clone.")

try:
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    else:
        local_changes = subprocess.check_output(
            ["git", "-C", str(REPO), "status", "--porcelain"],
            text=True,
        ).strip()
        if local_changes:
            raise RuntimeError(
                "The setup checkout has local changes. Use a clean Colab runtime before updating it."
            )
        subprocess.run(
            ["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"],
            check=True,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        "Could not install the repository's main branch. Check the Colab connection and try Setup again."
    ) from error

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    cwd=REPO,
    check=True,
)

source_path = REPO / "src"
if not (source_path / "boundary_pinn" / "__init__.py").is_file():
    raise FileNotFoundError(f"Package source not found in {source_path}.")
if str(source_path) not in sys.path:
    sys.path.insert(0, str(source_path))

for module_name in list(sys.modules):
    if module_name == "boundary_pinn" or module_name.startswith("boundary_pinn."):
        del sys.modules[module_name]
importlib.invalidate_caches()

import boundary_pinn
print("Imported boundary_pinn from:", boundary_pinn.__file__)
revision = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

test_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONUNBUFFERED": "1"}
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    check=True,
    env=test_env,
)
print("Setup and CPU tests passed. Code revision:", revision)


## 4. CPU preflight: test the packaged runner

Run this after Setup passes. It runs all five methods for one epoch using the repository's tiny preset, default model and samplers, one fixed Monte Carlo anchor, eight short paths, and coarse finite-difference grids. This checks that the installed runner works; it does not run the model or sampler code you edit in this notebook. Section 11 runs a small smoke check of your model, sampler, optimizer, and scheduler before the longer CPU preparation.

Outputs go under `/content/periodic_committor_preflight`, temporary Colab storage. The cell sets a flag only after it completes; preparation and training require this check to pass.

In [ ]:
# CPU only: run the packaged preset in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The full CPU experiment preparation is now available.')

## 5. Boundary conditions and the five trial functions

For each boundary set $j\in\{A,B\}$, the code builds a smooth level-set function $\omega_j$ and a normalized distance-like function $d_j$:

$$
\omega_j(x)=4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)-r^2,
\qquad
d_j(x)=\frac{\omega_j(x)}
{\sqrt{\omega_j(x)^2+\lvert\nabla\omega_j(x)\rvert^2}}.
$$

Here $c_j$ is the center of set $j$. The boundary is where $\omega_j=0$; in the exterior domain, both $d_A$ and $d_B$ are positive. Define the smooth lift $g$ and the two-boundary factor $\rho$ by

$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},
\qquad
\rho(x)=\frac{d_A(x)d_B(x)}
{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$

On $\omega_j=0$, where $\nabla\omega_j\ne0$, differentiation gives $\nabla d_j=\nabla\omega_j/|\nabla\omega_j|$, hence $|\nabla d_j|=1$. This is the first-order normalization: near its own boundary, $d_j$ changes at unit rate in the normal direction.

To see why the two-set factor is also normalized, write $F(a,b)=ab/\sqrt{a^2+b^2}$, so $\rho=F(d_A,d_B)$. On $\partial A$, the sets are disjoint and $d_B>0$. Since $\partial_aF(0,d_B)=1$ and $\partial_bF(0,d_B)=0$, we get $\nabla\rho=\nabla d_A$ and $|\nabla\rho|=1$. The same calculation gives $\nabla\rho=\nabla d_B$ and unit gradient on $\partial B$. The outward normal of the exterior domain points into a removed set, so $\partial_{n_D}\rho=-1$ on either boundary.

A plain product $d_A d_B$ also vanishes on both boundaries, but its normal derivative is scaled by the distance to the other set. The normalized $\rho$ removes that scale. Let $s$ be distance measured into $D$ from either boundary. Since $\partial_{n_D}\rho=-1$, $\rho=s+O(s^2)$ there. Thus $\rho^\alpha N_\theta$ behaves like $s^\alpha N_\theta$: for $\alpha=1$, its normal derivative at the boundary is finite and equals $-N_\theta$; for $\alpha>1$, its first normal derivative vanishes; for $\alpha<1$, it can be singular when $N_\theta$ is nonzero on the boundary. Thus $g=0$ on $\partial A$, $g=1$ on $\partial B$, and $\rho=0$ on both. The three exact-boundary trial functions use

$$
q_\theta(x)=g(x)+\rho(x)^\alpha N_\theta(x).
$$

Boundary-adapted uses $\alpha=1$, over-enforced uses $\alpha=3/2$, and under-enforced uses $\alpha=1/2$. Each satisfies both Dirichlet values by construction. The network output is periodic because the network receives sine and cosine coordinates. For background on the standard PINN residual and exact distance-function boundary enforcement, see [Raissi et al. (2019)](https://doi.org/10.1016/j.jcp.2018.10.045) and [Sukumar and Srivastava (2022)](https://doi.org/10.1016/j.cma.2021.114333); the exponent variants listed here are the specific cases in this experiment.

## 6. Which loss belongs to which method?

Every method uses the same interior collocation points and the same Monte Carlo anchor labels within a comparison. Write the interior residual loss and shared solution-value data loss as

$$
\widehat J_{\rm pde}(\theta)
=\frac{1}{N_D}\sum_{i=1}^{N_D}
\left|\mathcal Lq_\theta(x_i)\right|^2,
\qquad
\widehat J_{\rm data}(\theta)
=\frac{1}{N_{\rm data}}\sum_{i=1}^{N_{\rm data}}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$

| Method | Trial function | Trained loss components | Boundary treatment |
|---|---|---|---|
| Standard PINN (`simple_pinn`) | $q_\theta=N_\theta$ | PDE, Monte Carlo data, ordinary boundary residual | Samples both boundaries and penalizes the Dirichlet residual. |
| Boundary-adapted | $g+\rho N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Over-enforced | $g+\rho^{3/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Under-enforced | $g+\rho^{1/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Variational/Nitsche | $q_\theta=N_\theta$ | Interior-plus-flux loss + Nitsche boundary loss + Monte Carlo data | Weakly imposes the boundary values using Nitsche's method. |

Only the standard PINN has the ordinary boundary-residual component:

$$
\widehat J_{\rm bc}
=\frac{1}{N_A}\sum_i q_\theta(x_i^A)^2
+\frac{1}{N_B}\sum_i\bigl(q_\theta(x_i^B)-1\bigr)^2.
$$

The boundary-adapted, over-enforced, and under-enforced methods have **no extra boundary penalty**. The variational method's stabilization term belongs to the Nitsche weak formulation; it is not applied to an exact-boundary ansatz. The adaptive objective and exact component names are defined in Section 8; the variational method is derived in Section 9. The main experiment guide links each equation to its implementation.

## 7. Monte Carlo labels: the slow CPU stage and its live progress

The committor also has a stochastic representation. Starting at anchor $z_i$, independent Euler–Maruyama trajectories approximate the Langevin dynamics:

$$
X_{n+1}
=\left[X_n-\nabla V(X_n)\Delta t
+\sqrt{2\varepsilon\Delta t}\,\xi_n\right]\pmod 1,
\qquad \xi_n\sim\mathcal N(0,I_2).
$$

This carries over the pointwise solution-value regularizer used in the mean-escape-time notebooks: here the target value is the committor probability, and it complements rather than replaces the PDE residual. Each trajectory stops on first entering $A$ or $B$, or when it reaches the configured step cap. The label is the fraction of resolved trajectories that hit $B$ first:

$$
\widehat q_i
=\frac{\#\{\text{resolved trajectories from }z_i\text{ that hit }B\text{ first}\}}
{\#\{\text{resolved trajectories from }z_i\}}.
$$

Capped, unresolved paths are **not** counted as hits of $A$. Training stops with an error if an anchor's resolved fraction is below the configured minimum. The reported binomial standard error is conditional on the resolved paths; if paths remain unresolved, the guide explains the possible censoring bias and time-step error.

**What you will see while it runs:** reports show the step cap, paths resolved to (A) and (B), paths still active, anchors meeting the configured resolution threshold, the minimum and mean resolved fractions, and elapsed time. They appear at the configured fraction of the step cap or after the configured time interval, whichever comes first. The final resolution summary appears before training. After generation, the next cell saves and displays a plot of anchor locations colored by their estimated committor values, plus each label with its estimated Monte Carlo standard error. All methods share the same anchors and labels in `monte_carlo_labels.npz` and `monte_carlo_labels.json`. Neural-network approaches to committor computation are discussed, for example, by [Khoo, Lu, and Ying (2018)](https://arxiv.org/abs/1802.10275) and [Song, Cameron, and Yang (2023)](https://arxiv.org/abs/2306.12268).

### Set the amount and accuracy of Monte Carlo label data

An **anchor** is one input location where the notebook estimates the committor. `N_ANCHORS` is the number of locations; `PATHS_PER_ANCHOR` is the number of independent simulated trajectories per location. More paths reduce sampling noise and take longer.

Each path uses Euler–Maruyama with step size `TIME_STEP` and stops after hitting A or B, or after `MAX_STEPS`; their product is the maximum simulated time. A capped path is unresolved, not counted as a hit of A. If an anchor’s resolved fraction is below `MINIMUM_RESOLVED_FRACTION`, preparation stops with an error rather than silently accepting an unreliable label. `MONTE_CARLO_SEED` controls this simulation separately from network training.

The path count, step cap, and time step must be positive; the minimum resolved fraction must lie between 0 and 1. `PROGRESS_INTERVAL_FRACTION` must be greater than 0 and no more than 1. It is a fraction: the default 0.01 means 1% of the step cap. Progress appears when either that fraction or the wall-clock interval is reached, whichever comes first. The cell shows live progress; `monte_carlo_labels.json` saves the completed run settings and resolution counts.


In [ ]:
# Monte Carlo label-generation settings.
N_ANCHORS = 32
PATHS_PER_ANCHOR = 256
TIME_STEP = 0.001
MAX_STEPS = 100000
MINIMUM_RESOLVED_FRACTION = 0.9
MONTE_CARLO_SEED = secrets.randbelow(2**31)  # Disposable code-check seed; saved runs draw a new one.
PROGRESS_INTERVAL_FRACTION = 0.01
PROGRESS_INTERVAL_SECONDS = 10


## 8. Adaptive weighting in the paper's notation

The paper denotes the total adaptive objective by $\widehat J$ and its active empirical loss components by $\widehat J_i$. For each component, $\lambda_i>0$ is a trainable scale. Its objective (Eq. (33)) is

$$
\widehat J(\theta,\{\lambda_i\}_{i\in\mathcal I})
=\sum_{i\in\mathcal I}\frac{1}{2\lambda_i^2}\widehat J_i(\theta)
+\sum_{i\in\mathcal I}\log\lambda_i.
$$

See [Tepakbong et al. (2026), Eq. (33)](https://arxiv.org/abs/2607.19167). The active set $\mathcal I$ follows the method being trained:

- Standard PINN (simple_pinn): $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm bc}$ is the sampled Dirichlet residual.
- Boundary-adapted, over-enforced, and under-enforced: $\{\mathrm{pde},\mathrm{data}\}$. Their boundary values are built into the ansatz, so there is no boundary-residual term.
- Variational/Nitsche: $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm pde}$ is the squared interior-plus-flux residual, and $\widehat J_{\rm bc}$ is the squared Nitsche boundary contribution (symmetric consistency plus stabilization). This is not the standard PINN's pointwise boundary residual.

For this committor, $\widehat J_{\rm pde}$ uses the residual $\mathcal Lq_\theta=0$, and $\widehat J_{\rm data}$ compares $q_\theta(z_i)$ with the simulated committor label $\widehat q_i$. The loss definitions are in Section 6 and the Nitsche split is derived in Section 9. In the training logs, the three optimized Nitsche components are pde_loss, boundary_loss, and data_loss. The separate Nitsche term mean squares and the full weak-residual mean square are diagnostics, not separately weighted losses.

**How the code's saved scale corresponds to $\lambda_i$.** The code stores each trainable scale in log form, $s_i=\log\sigma_i$, and multiplies each raw loss by $e^{-2s_i}$. The paper's parameter is $\lambda_i=e^{s_i}/\sqrt{2}$, so $e^{-2s_i}=1/(2\lambda_i^2)$. Thus each weighted component has exactly the paper's coefficient. The code's total objective differs from the paper's displayed objective only by the constant $|\mathcal I|\log(2)/2$, which has zero gradient and does not change training. The raw loss components and effective weights are logged separately. The trainable-scale construction follows [Kendall, Gal, and Cipolla (2018)](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).

The notebook's `training.log_interval_seconds` setting controls the wall-clock interval between live console reports. The first and final epochs always print; each report includes the epoch, elapsed time, ETA, objective, and active raw loss components. Regardless of that interval, every epoch is written to `metrics.jsonl` with raw components, weighted components, effective weights, learning rate, and timing. Thus `losses.png` shows the full per-epoch history. `training.checkpoint_every` separately controls restartable checkpoints. Final raw loss components, the best epoch, and the objective are saved in `final_training_metrics.json` and `summary.json`.

Each trainable log scale starts at `training.adaptive_loss_log_scale_init` (zero in the pilot), then is optimized jointly with the network using the configured optimizer and learning-rate schedule. A zero starting value makes every raw loss coefficient initially one in the code objective.


### Choose the training duration and PDE sampling budget

One epoch is one optimizer update for each selected method, using newly sampled interior points. There is no early stopping. `EPOCHS` is the update budget per method; it and the point counts must be positive integers. The PDE collocation points are sampled in the domain at each update. `BOUNDARY_POINTS` are sampled on each boundary for the standard PINN’s boundary penalty; exact-boundary methods do not use that penalty, and Nitsche has separate quadrature controls below.


In [ ]:
# Training settings; one epoch is one optimizer update per selected method.
EPOCHS = 6000
COLLOCATION_POINTS = 1024
BOUNDARY_POINTS = 256


### Choose an optimizer in PyTorch

The next cell constructs the optimizer directly. Replace `torch.optim.Adam(...)` with any optimizer whose `.step()` works with this training loop. You can set its arguments using PyTorch's usual keyword parameters.

In [ ]:
LEARNING_RATE = 0.001
OPTIMIZER_BETAS = (0.9, 0.999)


def make_optimizer(parameters):
    return torch.optim.Adam(parameters, lr=LEARNING_RATE, betas=OPTIMIZER_BETAS)


### Choose a learning-rate schedule in PyTorch

This is also a normal constructor call. Replace it with another `torch.optim.lr_scheduler` class if you want. The training loop calls `scheduler.step()` once after each optimizer update.

In [ ]:
SCHEDULE_GAMMA = 0.9999


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=SCHEDULE_GAMMA)


### Replace the per-epoch training update (optional)

The packaged default computes the selected paper-method losses and performs one optimizer and scheduler step per epoch. Set `TRAINING_STEP` to a callback to replace that update completely. The callback receives the model, problem, full configuration, epoch, optimizer, scheduler, adaptive loss scales, Monte Carlo data, device, dtype, and interior sampler. It must perform its own updates and return a mapping with a finite scalar `objective`; any additional scalar values are written to `metrics.jsonl`. Return `"stop_training": True` to end after a durable checkpoint. This lets you change the objective, sampling schedule, number of updates, gradient handling, scheduler timing, stopping rule, and logged diagnostics.

The runnable example below uses only the strong PDE residual. It is an API example, not one of the paper formulations. Leave `TRAINING_STEP = None` to use the built-in method losses. A changed callback is recorded by name and code fingerprint; resume only with the same callback code.


In [ ]:
# Optional complete replacement for the built-in one-update-per-epoch logic.
def residual_only_training_step(
    *, model, problem, config, epoch, optimizer, scheduler, loss_scales,
    mc_data, device, dtype, interior_sampler,
):
    sampler = interior_sampler or sample_training_points
    points = sampler(
        problem, int(config.training["collocation_points"]),
        device=device, dtype=dtype,
    )
    objective = problem.residual(model, points).square().mean()
    optimizer.zero_grad(set_to_none=True)
    objective.backward()
    optimizer.step()
    scheduler.step()
    return {"objective": objective.detach(), "pde_loss": objective.detach()}

# Keep None for the built-in five-method objectives.
TRAINING_STEP = None
# To use the residual-only example instead, assign: TRAINING_STEP = residual_only_training_step


### Progress and restart checkpoints

Live reports show the epoch, elapsed time, ETA, objective, and active raw loss components. Every epoch is saved in `metrics.jsonl`; periodic checkpoints allow a stopped training run to resume.

In [ ]:
ADAPTIVE_LOSS_LOG_SCALE_INIT = 0.0
LOG_INTERVAL_SECONDS = 30
CHECKPOINT_EVERY = 100


## 9. Variational/Nitsche method: how its two trained losses are formed

Because the drift is a gradient, set $m(x)=e^{-V(x)/\varepsilon}$. The strong committor equation is equivalent to

$$
\mathcal Lq=0
\quad\Longleftrightarrow\quad
\nabla\cdot\left(\varepsilon m\nabla q\right)=0.
$$

For periodic Gaussian test functions $\phi_j$ centered at $c_j$, let $n_D$ be the outward normal of the exterior domain $D$, pointing into the removed sets:

$$
\phi_j(x)=\exp\!\left[
-\frac{4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)}{2\ell^2}
\right].
$$

The interior-plus-flux contribution is

$$
A_j=
\varepsilon\int_D m\nabla q_\theta\cdot\nabla\phi_j\,dx
-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds.
$$

The symmetric consistency term and Nitsche stabilization moment are

$$
C_j=-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds,
\qquad
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds,
$$

where $g=0$ on $\partial A$ and $g=1$ on $\partial B$. Together they form the Nitsche boundary contribution $B_j=C_j+P_j$. The full weak residual is $R_j=A_j+B_j$.

The paper's VPINN loss split is

$$
\widehat J_{\rm pde}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}A_j^2,
\qquad
\widehat J_{\rm bc}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}B_j^2.
$$

These two losses are optimized with the common committor data term $\widehat J_{\rm data}$. In the training log they are named pde_loss, boundary_loss, and data_loss. Nitsche stabilization belongs inside the boundary component; it is not an extra loss on an exact-boundary ansatz or the standard PINN's pointwise boundary residual. The separate volume, flux, consistency, stabilization, and full-residual mean squares are diagnostics, not additional adaptive losses. This follows [Liao and Ming (2021)](https://doi.org/10.4208/cicp.OA-2020-0219) and [Berrone et al. (2023)](https://doi.org/10.1016/j.heliyon.2023.e18820).

**Check the equations in code:** the PDE and boundary geometry are in src/boundary_pinn/problems/committor/periodic_two_well.py; Monte Carlo and weak-form integrals are in paper_methods.py; ansatzes, active loss sets, adaptive scales, and epoch logs are in training.py; orchestration and saved error tables are in comparison.py.

### Set Nitsche integration controls (used only for Variational/Nitsche)

`NITSCHE_VOLUME_POINTS` controls interior quadrature, and `NITSCHE_BOUNDARY_POINTS` controls boundary quadrature. `NITSCHE_TEST_FUNCTIONS` is the number of Gaussian test functions; `TEST_FUNCTION_LENGTH` is their width. `STABILIZATION_GAMMA` controls the strength of the Nitsche stabilization term, and `BOUNDARY_SCALE_H` is the characteristic boundary length scale in that term. The test-function length and `h` are measured in the unit-torus coordinate scale. These settings affect only the Nitsche formulation.


In [ ]:
# Used only when "variational_nitsche" is in METHODS_TO_RUN.
NITSCHE_VOLUME_POINTS = 1024
NITSCHE_TEST_FUNCTIONS = 32
NITSCHE_BOUNDARY_POINTS = 512
TEST_FUNCTION_LENGTH = 0.12
STABILIZATION_GAMMA = 10.0
BOUNDARY_SCALE_H = 0.03


## 10. Evaluation batch size

After training, the notebook compares the network's predictions with the finite-difference solution prepared earlier. It does this at 65,536 grid points (256 points in each coordinate).

The network processes 8,192 points at a time, so it takes 8 calls to cover the grid. The notebook then compares all predictions with the finite-difference values and calculates the errors.

`EVALUATION_BATCH_SIZE` sets the number of points in each call. It affects only this final comparison, not training or the network. Leave it at 8192 unless evaluation runs out of memory; then try 4096.

In [ ]:
REFERENCE_GRID_SIZES = [128, 256]
EVALUATION_BATCH_SIZE = 8192  # Points processed at a time for the final error calculation.


## 11. Prepare the run

After Setup and the CPU preflight pass, run both following cells. The first records the settings. The second checks your sampler, model, PDE residual, optimizer, and scheduler; if `TRAINING_STEP` is set, it also checks one disposable epoch per selected method. Temporary files are removed automatically. This is not experiment training. Rerun the check after changing code or settings.

In [ ]:
# Serialization for the separate CPU and training stages. Edit the Python choices above.
import json
from pathlib import Path

problem_settings = {
    "a": POTENTIAL_A,
    "b": POTENTIAL_B,
    "epsilon": DIFFUSION_EPSILON,
    "source_center": list(SOURCE_CENTER),
    "target_center": list(TARGET_CENTER),
    "set_radius": COMMON_CHORDAL_RADIUS,
}
monte_carlo_settings = {
    "n_anchors": N_ANCHORS,
    "paths_per_anchor": PATHS_PER_ANCHOR,
    "dt": TIME_STEP,
    "max_steps": MAX_STEPS,
    "seed": MONTE_CARLO_SEED,
    "minimum_resolved_fraction": MINIMUM_RESOLVED_FRACTION,
    "progress_interval_fraction": PROGRESS_INTERVAL_FRACTION,
    "progress_interval_seconds": PROGRESS_INTERVAL_SECONDS,
}
nitsche_settings = {
    "volume_points": NITSCHE_VOLUME_POINTS,
    "test_functions": NITSCHE_TEST_FUNCTIONS,
    "boundary_points": NITSCHE_BOUNDARY_POINTS,
    "length_scale": TEST_FUNCTION_LENGTH,
    "gamma": STABILIZATION_GAMMA,
    "h": BOUNDARY_SCALE_H,
}
training_settings = {
    "epochs": EPOCHS,
    "collocation_points": COLLOCATION_POINTS,
    "boundary_points": BOUNDARY_POINTS,
    "learning_rate": LEARNING_RATE,
    "checkpoint_every": CHECKPOINT_EVERY,
    "log_interval_seconds": LOG_INTERVAL_SECONDS,
    "adaptive_loss_log_scale_init": ADAPTIVE_LOSS_LOG_SCALE_INIT,
    "optimizer": {"name": "adam", "betas": list(OPTIMIZER_BETAS)},
    "scheduler": {"name": "exponential", "gamma": SCHEDULE_GAMMA},
    "monte_carlo": monte_carlo_settings,
    "nitsche": nitsche_settings,
}
RUN_CONFIG = {
    "seed": SEED,
    "device": DEVICE,
    "dtype": DTYPE,
    "methods": METHODS_TO_RUN,
    "problem": problem_settings,
    "training": training_settings,
    "reference_grid_sizes": REFERENCE_GRID_SIZES,
    "evaluation_batch_size": EVALUATION_BATCH_SIZE,
}

RUN_CONFIG_PATH = Path("/content/periodic_committor_selected_config.json")
RUN_CONFIG_PATH.write_text(
    json.dumps(RUN_CONFIG, indent=2) + "\n",
    encoding="utf-8",
)
print("Methods:", ", ".join(METHODS_TO_RUN))
print("Problem:", problem_settings)
print("Device / dtype / code-check seed:", DEVICE, DTYPE, SEED)
print("Training settings:", training_settings)
print("Saved scalar settings:", RUN_CONFIG_PATH)


## 12. CPU preparation: reference solution and Monte Carlo labels

Run this after Setup, the CPU preflight, and the section 11 code check. This stage uses CPU whether Colab is running on CPU or GPU; keep the selected runtime attached. It does not train a network. It solves the finite-difference reference on `REFERENCE_GRID_SIZES` (default: `[128, 256]`), then estimates committor labels by simulating trajectories from the sampled anchors. It uses the selected problem, methods, and interior sampler. The network, optimizer, and scheduler are used later during training.

With the defaults above, Monte Carlo simulates 32 anchors × 256 paths = 8,192 trajectories, with `TIME_STEP = 0.001` and a cap of 100,000 steps per path. This is usually the slow part. Progress reports show the step cap, paths resolved in A and B, active paths, how many anchors meet the resolution threshold, the minimum and mean resolved fractions, and elapsed time. Reports appear at the configured fraction of the step cap or time interval, whichever comes first (defaults: 1,000 steps or 10 seconds).

Each anchor must meet `MINIMUM_RESOLVED_FRACTION` (default: 0.9). If it does not, preparation stops with an error; unresolved paths are not counted as hits of A. Reducing `N_ANCHORS` or `PATHS_PER_ANCHOR` makes the run faster but noisier. Lowering `MAX_STEPS` can leave more paths unresolved.

Preparation creates a unique results folder and stores its path in `RUN_ROOT`. The next training cell reads that variable directly; there is no path to copy or paste. The folder stores the preparation settings, code revision, reference, labels, progress metadata, and anchor plot. Keep the problem, reference-grid sizes, Monte Carlo settings, and sampler unchanged when using those labels. Before the first training run, you can change the architecture, initialization, methods, training budget, optimizer, or schedule. Rerun section 11, then train from the same folder. The preparation seed and Monte Carlo labels are reused; training settings are saved separately. Save this notebook copy with the run because the model and custom Python code are defined here.

In [ ]:
# Run CPU preparation and keep live progress in this cell.
from datetime import datetime
from pathlib import Path
import json
import subprocess
import torch
from IPython.display import Image, display
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('Run Setup and pass the packaged CPU preflight in section 4 first.')
if not globals().get('NOTEBOOK_CODE_SMOKE_PASSED', False):
    raise RuntimeError('Run section 11 to smoke-check your model, sampler, optimizer, and scheduler first.')

torch.set_num_threads(2)
# Each new preparation folder gets its own fresh seed pair.
SEED = secrets.randbelow(2**31)
MONTE_CARLO_SEED = secrets.randbelow(2**31)
RUN_CONFIG["seed"] = SEED
RUN_CONFIG["training"]["monte_carlo"]["seed"] = MONTE_CARLO_SEED
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + "\n", encoding="utf-8")

run_name = (
    'periodic_committor_'
    + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '_'
    + secrets.token_hex(4)
)
output_parent = run_output_parent()
output_parent.mkdir(parents=True, exist_ok=True)
RUN_ROOT = str(output_parent / run_name)
Path(RUN_ROOT).mkdir(parents=False, exist_ok=False)
model_settings_path = Path(RUN_ROOT, "notebook_model_settings.json")
model_settings_path.write_text(
    json.dumps(NOTEBOOK_MODEL_SETTINGS, indent=2) + "\n",
    encoding="utf-8",
)
print(f'Training seed: {SEED}; Monte Carlo seed: {MONTE_CARLO_SEED}', flush=True)
print('CPU preparation folder created; training will use it automatically:', RUN_ROOT, flush=True)

run_periodic_comparison(
    RUN_CONFIG_PATH,
    Path(RUN_ROOT),
    prepare_only=True,
    interior_sampler=sample_training_points,
)
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
preflight_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('preparation_git_revision') != preflight_revision:
    raise RuntimeError(
        "Preflight and preparation use different code revisions. Repeat both on one revision."
    )
prep_record['cpu_preflight_passed'] = True
prep_record['cpu_preflight_git_revision'] = preflight_revision
prep_path.write_text(json.dumps(prep_record, indent=2) + '\n', encoding='utf-8')
print('CPU preparation is complete. No model training has run.', flush=True)
anchor_plot = Path(RUN_ROOT) / 'monte_carlo_anchors.png'
if anchor_plot.is_file():
    print('Monte Carlo anchor plot (inspect before training):', anchor_plot)
    display(Image(filename=str(anchor_plot)))


## 13. Training

The saved `DEVICE` setting chooses where training runs. If it is `"cuda"`, choose a GPU runtime before Setup and keep it connected; if it is `"cpu"`, choose a CPU runtime. CPU preparation has already completed in this runtime, so no restart or cell rerun is needed.

Run the next cell. It uses the preparation folder stored in `RUN_ROOT`, loads the reference and labels, and trains with the current methods, epoch budget, point counts, model, optimizer, and schedule. It checks that the saved problem, reference grids, and Monte Carlo labels match; compatible training-setting changes do not repeat CPU preparation. It reports losses at the configured interval and saves checkpoints in that same folder.

If Colab has reset, the cell looks for an unfinished prepared run. It selects the only match automatically; if there is more than one, choose its number from the short list. No folder path needs to be copied or pasted.

In [ ]:
# Train from the saved CPU preparation with the model, sampler, optimizer, and schedule above.
from pathlib import Path
import json
import torch
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

if not globals().get('NOTEBOOK_CODE_SMOKE_PASSED', False):
    raise RuntimeError('Rerun section 11 after defining or editing your model and callbacks.')

run_root = resolve_run_root(
    "cpu_preparation.json",
    purpose="train",
    unfinished_only=True,
)
RUN_ROOT = str(run_root)
model_settings_path = Path(RUN_ROOT, "notebook_model_settings.json")
checkpoint_names = ("last.pt", "best.pt", "final.pt")
has_training_checkpoints = any(
    path.is_file()
    for name in checkpoint_names
    for path in Path(RUN_ROOT).glob(f"*/{name}")
)
if has_training_checkpoints:
    if not model_settings_path.is_file():
        raise RuntimeError("This run has checkpoints but no saved model settings.")
    saved_model_settings = json.loads(model_settings_path.read_text(encoding="utf-8"))
    if saved_model_settings != NOTEBOOK_MODEL_SETTINGS:
        raise RuntimeError("Restore the saved model settings before resuming.")
    raise RuntimeError(
        "This folder already has training checkpoints. Use the resume section or start a new preparation."
    )
model_settings_path.write_text(
    json.dumps(NOTEBOOK_MODEL_SETTINGS, indent=2) + "\n",
    encoding="utf-8",
)

prep_path = Path(RUN_ROOT, "cpu_preparation.json")
if not prep_path.is_file():
    raise FileNotFoundError(f'No completed CPU preparation was found in {RUN_ROOT}.')
prepared_config_path = Path(RUN_ROOT, "comparison_config.json")
if not prepared_config_path.is_file():
    raise FileNotFoundError(f"Saved preparation settings are missing: {prepared_config_path}")
prepared_config = json.loads(prepared_config_path.read_text(encoding="utf-8"))

# Keep the fresh seeds recorded with this preparation after a runtime reset.
RUN_CONFIG["seed"] = prepared_config["seed"]
RUN_CONFIG["training"]["monte_carlo"]["seed"] = (
    prepared_config["training"]["monte_carlo"]["seed"]
)
SEED = int(RUN_CONFIG["seed"])
MONTE_CARLO_SEED = int(RUN_CONFIG["training"]["monte_carlo"]["seed"])
RUN_CONFIG_PATH.write_text(
    json.dumps(RUN_CONFIG, indent=2) + "\n",
    encoding="utf-8",
)
config_path = RUN_CONFIG_PATH
selected_config = json.loads(config_path.read_text(encoding="utf-8"))
training_device = str(selected_config["device"])
if training_device.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError(
        "This run selects CUDA, but no GPU is available in this runtime. "
        "Choose a GPU runtime, rerun Setup and the notebook code cells, then train. "
        "Do not repeat the completed CPU preparation."
    )
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
if (
    prep_record.get("cpu_preflight_passed") is not True
    or prep_record.get("cpu_preflight_git_revision")
    != prep_record.get("preparation_git_revision")
):
    raise RuntimeError("The saved CPU preparation was not paired with its passing preflight.")

print(f'{training_device.upper()} training outputs will be saved in:', RUN_ROOT, flush=True)
run_periodic_comparison(
    config_path,
    Path(RUN_ROOT),
    train_prepared=True,
    model_factory=PeriodicCommittorNetwork,
    interior_sampler=sample_training_points,
    optimizer_factory=make_optimizer,
    scheduler_factory=make_scheduler,
    training_step=TRAINING_STEP,
)


## 14. Inspect the completed run

The run folder contains `evaluation_arrays.npz` with the finite-difference grid/reference, masks, and each method’s prediction, signed error, and absolute error. Every method’s signed-error panel uses the same symmetric color limits; `summary.json` and the NPZ record those limits. `run_manifest.json` lists all artifacts and marks the comparison complete only after the required files exist. With local storage, this cell downloads a ZIP of the complete run.

Run after training. The cell uses `RUN_ROOT` from this notebook session. After a runtime reset, it finds saved completed runs automatically; if several are available, choose one by its number. It reads saved files only and does not train. It displays the finite-difference grid check; absolute and relative $L^2$, $H^1$, $H^1$-seminorm, $H^2$, and $H^2$-seminorm errors; final raw loss components; and saved plots.

Errors use the finest configured finite-difference solution. Grid-to-grid differences estimate reference-resolution sensitivity, not an error bound. Figures are previews; arrays, metrics, and checkpoints are saved in the run folder.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

run_root = resolve_run_root("summary.json", purpose="inspect completed results")
RUN_ROOT = str(run_root)

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run the evaluation section to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

custom_training_rows = {}
for method in method_summaries:
    method_dir = Path(RUN_ROOT) / method
    method_metadata_path = method_dir / 'metadata.json'
    method_metadata = json.loads(method_metadata_path.read_text()) if method_metadata_path.is_file() else {}
    training_hook = method_metadata.get('runtime_hooks', {}).get('training_step', 'repository training step')
    if training_hook != 'repository training step':
        final_metrics = method_summaries[method].get('final_training_metrics', {}).get('final_epoch_metrics', {})
        if not final_metrics:
            metrics_path = method_dir / 'metrics.jsonl'
            if metrics_path.is_file():
                rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
                if rows:
                    final_metrics = {
                        key: value for key, value in rows[-1].items()
                        if key not in {'epoch', 'loss', 'session_elapsed_seconds',
                                       'session_epochs_per_second', 'estimated_remaining_seconds'}
                    }
        custom_training_rows[method] = final_metrics
if custom_training_rows:
    print('Final values returned by the custom training-step callback:')
    display(pd.DataFrame(custom_training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

if not SAVE_TO_DRIVE and DOWNLOAD_LOCAL_RESULTS:
    import shutil
    from google.colab import files

    archive_base = Path(RUN_ROOT).parent / f"{Path(RUN_ROOT).name}_archive"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
    print("Download the complete run archive:", archive_path)
    files.download(archive_path)

## 15. Re-evaluate saved checkpoints on CPU

This separate cell reloads each method's `best.pt` checkpoint and recomputes errors against the saved finite-difference reference. It uses CPU only, does not retrain, and does not overwrite the original `summary.json`. It displays the Sobolev errors and final raw training-loss components and writes `saved_model_evaluation.json` to the same run folder.

In the same runtime, it uses the current `RUN_ROOT`. After a runtime reset, it finds saved runs automatically; if there are several, choose one by its number. Run the model and initialization cells first. Checkpoints contain weights, not your custom Python class, so use the same model code that created them.

In [ ]:
# Re-evaluate saved best.pt checkpoints on CPU; this does not train.
from pathlib import Path
import json
from IPython.display import display
import pandas as pd
from boundary_pinn.experiments.committor.periodic_two_well.comparison import evaluate_saved_models

run_root = resolve_run_root("summary.json", purpose="re-evaluate saved checkpoints")
RUN_ROOT = str(run_root)

model_settings_path = Path(RUN_ROOT, 'notebook_model_settings.json')
if model_settings_path.is_file():
    saved_model_settings = json.loads(model_settings_path.read_text(encoding='utf-8'))
    if saved_model_settings != NOTEBOOK_MODEL_SETTINGS:
        raise RuntimeError('Load the architecture and initialization recorded for this run before evaluation.')
elif NOTEBOOK_MODEL_SETTINGS != {
    'input_features': ['sin(2*pi*x1)', 'sin(2*pi*x2)', 'cos(2*pi*x1)', 'cos(2*pi*x2)'],
    'width': 64, 'depth': 2, 'activation': 'tanh', 'initialization': 'pytorch',
    'initialization_gain': 1.0, 'bias_initialization': 'pytorch',
    'bias_initialization_value': 0.0, 'use_bias': True,
}:
    raise RuntimeError(
        "This older run has no saved model settings; restore the original default network."
    )
evaluate_saved_models(Path(RUN_ROOT), model_factory=PeriodicCommittorNetwork)
report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel', 'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel', 'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])
print('Full saved report:', report_path)


## 16. Resume an interrupted run

Use this only when training stopped before all selected methods finished. A run saved with `device: cuda` needs a GPU runtime; a run saved with `device: cpu` uses CPU. If your current runtime already matches, keep it. If you change runtimes, Colab resets the live Python session, so rerun Setup and the notebook's code cells. The cell finds the unfinished prepared run automatically; if there is more than one, choose its number from a short list. It resumes from each method's `last.pt` and does not repeat preparation.

It checks the prepared-data format, saved objective version, callback names, and code fingerprints. It cannot recover Python source from notebook cells. Resume only with the exact model, sampler, optimizer, scheduler, and training-step code used before; use a new preparation for changed settings.

For a new experiment, return to the preparation section to create a fresh folder. Set `RESUME_INTERRUPTED_RUN = True` in the next cell only for an interrupted run.

In [ ]:
# Set True only to resume an interrupted training run.
RESUME_INTERRUPTED_RUN = False

if not RESUME_INTERRUPTED_RUN:
    print("Resume skipped. Set RESUME_INTERRUPTED_RUN = True to continue an interrupted run.")
else:
    from pathlib import Path
    import json
    import torch
    from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

    if not globals().get("NOTEBOOK_CODE_SMOKE_PASSED", False):
        raise RuntimeError("Rerun section 11 after defining or editing your model and callbacks.")

    run_root = resolve_run_root(
        "cpu_preparation.json",
        purpose="resume",
        unfinished_only=True,
    )
    RUN_ROOT = str(run_root)

    prep_path = Path(RUN_ROOT, "cpu_preparation.json")
    if not prep_path.is_file():
        raise FileNotFoundError("That folder has no completed CPU preparation.")
    prep_record = json.loads(prep_path.read_text(encoding="utf-8"))
    config_path = Path(RUN_ROOT, "comparison_config.json")
    if not config_path.is_file():
        raise FileNotFoundError(f"The run folder is missing its saved configuration: {RUN_ROOT}")
    selected_config = json.loads(config_path.read_text(encoding="utf-8"))

    current_config = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    current_config["seed"] = selected_config["seed"]
    current_config["training"]["monte_carlo"]["seed"] = (
        selected_config["training"]["monte_carlo"]["seed"]
    )
    if current_config != selected_config:
        raise RuntimeError(
            "Settings changed since training started. Restore them before resuming."
        )

    RUN_CONFIG = selected_config
    SEED = int(RUN_CONFIG["seed"])
    MONTE_CARLO_SEED = int(RUN_CONFIG["training"]["monte_carlo"]["seed"])
    RUN_CONFIG_PATH.write_text(
        json.dumps(RUN_CONFIG, indent=2) + "\n",
        encoding="utf-8",
    )

    training_device = str(selected_config["device"])
    if training_device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(
            "This saved run requires a GPU runtime. Choose GPU, rerun Setup and the "
            "notebook code cells, then resume. Do not repeat CPU preparation."
        )
    if (
        prep_record.get("cpu_preflight_passed") is not True
        or prep_record.get("cpu_preflight_git_revision")
        != prep_record.get("preparation_git_revision")
    ):
        raise RuntimeError("The saved CPU preparation was not paired with its passing preflight.")

    model_settings_path = Path(RUN_ROOT, "notebook_model_settings.json")
    saved_model_settings = json.loads(model_settings_path.read_text(encoding="utf-8"))
    if saved_model_settings != NOTEBOOK_MODEL_SETTINGS:
        raise RuntimeError("Restore the model settings used by this run before resuming.")

    run_periodic_comparison(
        config_path,
        Path(RUN_ROOT),
        train_prepared=True,
        resume=True,
        model_factory=PeriodicCommittorNetwork,
        interior_sampler=sample_training_points,
        optimizer_factory=make_optimizer,
        scheduler_factory=make_scheduler,
        training_step=TRAINING_STEP,
    )
    print(f"The interrupted {training_device.upper()} run has resumed and completed.")


## References

- Y. Liao and P. Ming, “Deep Nitsche Method: Deep Ritz Method with Essential Boundary Conditions,” *Communications in Computational Physics* 29(5) (2021), 1365–1384. [doi:10.4208/cicp.OA-2020-0219](https://doi.org/10.4208/cicp.OA-2020-0219).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- A. Kendall, Y. Gal, and R. Cipolla, “Multi-task learning using uncertainty to weigh losses for scene geometry and semantics,” *CVPR* (2018). [paper](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).